# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - LSTM Model (Depreciated)

### Written by Krupam Patel
### Written in Python 3.11.14

###### This file contains the code for the LSTM for the wind speeds to be predicted.

In [60]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [61]:
TARGET = "Direction (A)"
WINDOW = 48

In [62]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CLEAN.csv')
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

In [63]:
direction_bins = cleancsv[TARGET].to_numpy()

# Validate before converting to integers.
if not np.isin(direction_bins, np.arange(16)).all():
    raise ValueError(
        "Expected integer direction bins 0–15. "
        f"Found: {np.unique(direction_bins)}"
    )
output_data = direction_bins.astype(np.int32)

In [64]:
#Convert data into Date time and create date filter
cleancsv["Date"] = (
    pd.to_datetime(cleancsv["Date"], errors="coerce")
    + pd.to_timedelta(cleancsv["Hr"], unit="h")
)

cleancsv = cleancsv.drop(columns="Hr")
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

# Display conflicting timestamps for inspection.
duplicate_mask = cleancsv["Date"].duplicated(keep=False)

if duplicate_mask.any():
    print(cleancsv.loc[duplicate_mask].to_string(index=False))
    raise ValueError(
        "Duplicate timestamps found. Review the rows printed above."
    )
# Confirm that 24 rows really represent 24 consecutive hours.
hour_steps = cleancsv["Date"].diff().iloc[1:]

if cleancsv["Date"].isna().any():
    raise ValueError("Fix invalid timestamps before training.")

if cleancsv["Date"].duplicated().any():
    raise ValueError("Resolve duplicate timestamps before training.")

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [65]:

# Show every row sharing a timestamp.
duplicates = cleancsv[
    cleancsv.duplicated(subset="Date", keep=False)
]

print("ROWS WITH DUPLICATE TIMESTAMPS:")
print(duplicates.to_string(index=False))

# Show intervals that are not exactly one hour.
time_check = pd.DataFrame({
    "previous_time": cleancsv["Date"].shift(1),
    "current_time": cleancsv["Date"],
    "difference": cleancsv["Date"].diff()
})

bad_intervals = time_check.iloc[1:]
bad_intervals = bad_intervals[
    bad_intervals["difference"] != pd.Timedelta(hours=1)
]

print("\nIRREGULAR TIME INTERVALS:")
print(bad_intervals.to_string(index=False))

print("\nINVALID TIMESTAMPS:", cleancsv["Date"].isna().sum())

ROWS WITH DUPLICATE TIMESTAMPS:
Empty DataFrame
Columns: [Date, Mainland Air Temp, Humidity (%), Direction (A), Wind Speed (A), Gusting, Atmospheric Pressure (IN), Precipitation Rate, Bay Temp, Salinity, LBI Air Temp, Ocean Temp, Onshore, upwelling_flag]
Index: []

IRREGULAR TIME INTERVALS:
Empty DataFrame
Columns: [previous_time, current_time, difference]
Index: []

INVALID TIMESTAMPS: 0


In [66]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8, 9])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [67]:
cleancsv = cleancsv.sort_values("Date").reset_index(drop=True)

# Direction target
direction_bins = cleancsv[TARGET].to_numpy()

if not np.isin(direction_bins, np.arange(16)).all():
    raise ValueError(
        "Expected direction bins 0–15. "
        f"Found: {np.unique(direction_bins)}"
    )

output_data = direction_bins.astype(np.int32)

# the input window ends before the target hour.
feature_cols = [
    column for column in cleancsv.columns
    if column != "Date"
]

dataset = cleancsv[feature_cols].to_numpy(dtype=np.float32)

if not np.isfinite(dataset).all():
    raise ValueError("Inputs contain missing or infinite values.")

In [68]:
#Length of training data
n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

n = len(cleancsv)
train_end = int(n * 0.80)
test_start = int(np.ceil(n * 0.90))

if not WINDOW < train_end < test_start < n:
    raise ValueError("Not enough observations for these splits.")

In [69]:
# Standardize inputs only, using training-period statistics.
scaler_x = StandardScaler()
scaler_x.fit(dataset[:train_end])
scaledx = scaler_x.transform(dataset)

# Only retain windows with consecutive hourly timestamps.
hourly_step = (
    cleancsv["Date"].diff() == pd.Timedelta(hours=1)
)

valid_target = (
    hourly_step
    .rolling(window=WINDOW, min_periods=WINDOW)
    .sum()
    .eq(WINDOW)
)

target_rows = np.flatnonzero(valid_target.to_numpy())

if len(target_rows) == 0:
    raise ValueError("No complete 24-hour forecast windows found.")

X = np.stack([
    scaledx[i - WINDOW:i]
    for i in target_rows
])

y = output_data[target_rows]

# Create masks
train_mask = target_rows < train_end
val_mask = (
    (target_rows >= train_end)
    & (target_rows < test_start)
)
test_mask = target_rows >= test_start

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test = X[test_mask]

test_rows = target_rows[test_mask]

for name, labels in [("Training", y_train), ("Validation", y_val), ("Testing", y[test_mask])]:
    if labels.size == 0:
        raise ValueError(f"{name} split has no valid windows.")

    if not np.isin(labels, np.arange(16)).all():
        raise ValueError(f"{name} labels must be 0–15.")

    print(name, "labels:", np.unique(labels))

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape)
print("Excluded windows:", (n - WINDOW) - len(target_rows))

Training labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Validation labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 13]
Testing labels: [ 0  1  2  3  4  5  6  7  8  9 10 12 13 14 15]
Training: (2141, 48, 13) (2141,)
Validation: (275, 48, 13) (275,)
Testing: (273, 48, 13)
Excluded windows: 0


In [70]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

model = keras.models.Sequential([
    keras.layers.Input(
        shape=(WINDOW, len(feature_cols))
    ),
    keras.layers.LSTM(32),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(16, activation="softmax")
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [71]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [72]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    shuffle=False,
    callbacks=[early_stopping]
)

Epoch 1/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1457 - loss: 2.6927 - val_accuracy: 0.5673 - val_loss: 2.2503
Epoch 2/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2592 - loss: 2.4362 - val_accuracy: 0.6545 - val_loss: 1.5836
Epoch 3/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3148 - loss: 2.2341 - val_accuracy: 0.6473 - val_loss: 1.3708
Epoch 4/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3358 - loss: 2.0719 - val_accuracy: 0.6327 - val_loss: 1.3040
Epoch 5/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3587 - loss: 1.9452 - val_accuracy: 0.6218 - val_loss: 1.2850
Epoch 6/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4003 - loss: 1.8433 - val_accuracy: 0.6364 - val_loss: 1.2100
Epoch 7/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4096 - loss: 1.7590 - val_accuracy: 0.6291 - val_loss: 1.2022
Epoch 8/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4311 - loss: 1.6877 - val_accuracy: 0.6509 - v

In [73]:
probabilities = model.predict(X_test)
prediction = np.argmax(probabilities, axis=1)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [74]:
y_test_true = output_data[test_rows]
naive_prediction = output_data[test_rows - 1]
# persustebce =
naive_prediction = output_data[test_rows -1]

def circular_mae(actual, predicted):
    difference = np.abs(np.asarray(actual).ravel() - np.asarray(predicted).ravel())

    #shorter distance around the compass
    return np.minimum(difference, 16 - difference).mean()

lstm_mae = circular_mae(y_test_true, prediction)
naive_mae = circular_mae(y_test_true, naive_prediction)

print(f"LSTM circular MAE: {lstm_mae:.3f} bins")
print(f"Persistence circular MAE: {naive_mae:.3f} bins")

print(f"LSTM angular MAE: {lstm_mae * 22.5:.2f} degrees")

# Check whether the model captures variation.
print(f"Observed standard deviation: {y_test_true.std():.3f}")
print(f"Predicted standard deviation: {prediction.std():.3f}")

first_target = test_rows[0]

print(
    "First input window:",
    cleancsv["Date"].iloc[first_target - WINDOW],
    "through",
    cleancsv["Date"].iloc[first_target - 1]
)

print(
    "First predicted hour:",
    cleancsv["Date"].iloc[first_target]
)


LSTM circular MAE: 1.029 bins
Persistence circular MAE: 0.725 bins
LSTM angular MAE: 23.16 degrees
Observed standard deviation: 4.009
Predicted standard deviation: 3.908
First input window: 2025-09-10 16:00:00 through 2025-09-12 15:00:00
First predicted hour: 2025-09-12 16:00:00


In [75]:
# Saving to csv
y_test_true = np.asarray(y_test_true).ravel()
prediction = np.asarray(prediction).ravel()
naive_prediction = np.asarray(naive_prediction).ravel()

dates = cleancsv["Date"].iloc[test_rows].to_numpy()

# Confirm that every column has one value per forecast hour.
lengths = {
    "Date": len(dates),
    "observed": len(y_test_true),
    "LSTM": len(prediction),
    "naive": len(naive_prediction)
}

if len(set(lengths.values())) != 1:
    raise ValueError(f"Column lengths do not match: {lengths}")

test_df = pd.DataFrame({
    "Date": cleancsv["Date"].iloc[test_rows].to_numpy(),
    "observed_direction_bin": y_test_true.ravel(),
    "predicted_direction_lstm": prediction.ravel(),
    "predicted_direction_naive": naive_prediction.ravel()
})

os.makedirs("CSV/predictions", exist_ok=True)

test_df.to_csv(
    "CSV/predictions/wind_direction_lstm_predictions.csv",
    index=False
)